In [ ]:
import os
from pathlib import Path
import random

dataset_dir = Path("/net/holy-isilon/ifs/rc_labs/ydu_lab/xczhang/workspace/SAILOR/env_repos/LIBERO/libero/datasets/libero_object_replay")
video_files = sorted(list(dataset_dir.rglob("*merge*.mp4")))

pairs = []

for video_file in video_files:
    if "pose" in str(video_file):
        continue
    demo_id = int(str(video_file).split("/")[-2].split("_")[-1])
    # if demo_id > 0 and demo_id <= 20:
    #     split = "training"
    # # elif demo_id <= 50 and demo_id > 10:
    # #     split = "training"
    # else:
    #     continue
    # split = 'validation'
    # split = "training"
    if random.random() < 0.95:
        split = "training"
    else:
        split = "validation"
    pose_file = str(video_file).replace(".mp4", "_pose.mp4")
    low_dim_file = str(video_file).replace("merged", "low_dim").replace(".mp4", ".npz")
    pairs.append({
        "video": video_file,
        "pose_video": Path(pose_file),
        "low_dim_data": Path(low_dim_file),
        "split": split
    })

print(f"Total {len(pairs)} video-pose pairs found.")


In [2]:
import pandas as pd 
import csv
import decord
import numpy as np

override_fps = 20

min_len = 81

def process_pair(pair):
    video_path = pair["video"]
    pose_video_path = pair["pose_video"]
    low_dim_data_path = pair["low_dim_data"]
    split = pair["split"]
   
    if not video_path.exists():
        print(f"Video file not found: {video_path}")
        return None
    if not pose_video_path.exists():
        print(f"Pose video file not found: {pose_video_path}")
        return None
    if not low_dim_data_path.exists():
        print(f"Low-dimensional data file not found: {low_dim_data_path}")
        return None

    try:
        vr = decord.VideoReader(str(video_path))
        n_frames = len(vr)
        width = vr[0].shape[1]
        height = vr[0].shape[0]
        del vr
    except Exception as e:
        print(f"Error loading video {video_path}: {e}")
        return None

    if n_frames < min_len:
        return None

    try:
        low_dim_data = np.load(low_dim_data_path)
        n_action_frames = low_dim_data['actions'].shape[0]
    except Exception as e:
        print(f"Error loading low-dimensional data file {low_dim_data_path}: {e}")
        return None

    try:
        vr_pose = decord.VideoReader(str(pose_video_path))
        n_pose_frames = len(vr_pose)
        del vr_pose
    except Exception as e:
        print(f"Error loading pose video {pose_video_path}: {e}")
        return None

    if n_frames != n_action_frames or n_frames != n_pose_frames:
        print(
            f"Frame count mismatch: {video_path} has {n_frames} frames, "
            f"but {low_dim_data_path} has {n_action_frames} frames."
            f" and {pose_video_path} has {n_pose_frames} frames."
        )
        return None

    return {
        "video_path": str(video_path.resolve()),
        "cond_path": str(pose_video_path.resolve()),
        "low_dim_path": str(low_dim_data_path.resolve()),
        "fps": override_fps,
        "n_frames": n_frames,
        "width": width,
        "height": height,
        "split": split,
        # "trim_start": 0,
        # "trim_end": 49,
        }

In [3]:
import os
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

max_workers = min(64, os.cpu_count() or 4)
records_parallel = []
with ThreadPoolExecutor(max_workers=max_workers) as ex:
    futures = {ex.submit(process_pair, pair): pair for pair in pairs}
    for f in tqdm(
        as_completed(futures),
        total=len(futures),
        desc="Building metadata (parallel)",
    ):
        rec = f.result()
        if rec is not None:
            records_parallel.append(rec)
records = records_parallel
import random
random.shuffle(records)

metadata_df = pd.DataFrame.from_records(records)
metadata_csv_path = dataset_dir / "metadata.csv"
metadata_df.to_csv(metadata_csv_path, index=False, quoting=csv.QUOTE_MINIMAL)
print(f"Metadata saved to {metadata_csv_path}")

check_df = pd.read_csv(metadata_csv_path)
print(f"Loaded metadata with {len(check_df)} entries.")
print(check_df['split'].value_counts())
print(check_df['width'].unique())
print(check_df['height'].unique())

Building metadata (parallel):   0%|              | 1/1410 [00:00<05:04,  4.63it/s]

Building metadata (parallel):  19%|██         | 264/1410 [00:01<00:03, 365.18it/s]

Low-dimensional data file not found: /net/holy-isilon/ifs/rc_labs/ydu_lab/xczhang/workspace/SAILOR/env_repos/LIBERO/libero/datasets/libero_object_replay/args_std_0.12_224_224_chunk21_pos_len81_hist40/pick_up_the_alphabet_soup_and_place_it_in_the_basket_demo/demo_32/low_dim_seg1.npz


Building metadata (parallel): 100%|██████████| 1410/1410 [00:03<00:00, 357.06it/s]


Metadata saved to /net/holy-isilon/ifs/rc_labs/ydu_lab/xczhang/workspace/SAILOR/env_repos/LIBERO/libero/datasets/libero_object_replay/metadata.csv
Loaded metadata with 1409 entries.
split
training      1131
validation     278
Name: count, dtype: int64
[448]
[448]


In [4]:
import pandas as pd

csv_files = [
    "/n/holylabs/ydu_lab/Lab/zhangxiangcheng/code/SAILOR/env_repos/LIBERO/libero/datasets/metadata/metadata_20260210_085632.csv", \
    "/net/holy-isilon/ifs/rc_labs/ydu_lab/xczhang/workspace/SAILOR/env_repos/LIBERO/libero/datasets/libero_90_replay/args_std_0.0_224_224_chunk21_len81_hist40/metadata.csv", \
    "/net/holy-isilon/ifs/rc_labs/ydu_lab/xczhang/workspace/SAILOR/env_repos/LIBERO/libero/datasets/libero_90_replay/args_std_0.13_224_224_chunk21_gripper_len81_hist40/metadata.csv", \
    "/net/holy-isilon/ifs/rc_labs/ydu_lab/xczhang/workspace/SAILOR/env_repos/LIBERO/libero/datasets/libero_90_replay/args_std_0.18_224_224_chunk21_gripper_pos_len81_hist40/metadata.csv", \
    "/net/holy-isilon/ifs/rc_labs/ydu_lab/xczhang/workspace/SAILOR/env_repos/LIBERO/libero/datasets/libero_90_replay/args_std_0.26_224_224_chunk21_pos_len81_hist40/metadata.csv", \
    "/net/holy-isilon/ifs/rc_labs/ydu_lab/xczhang/workspace/SAILOR/env_repos/LIBERO/libero/datasets/libero_10_replay/args_std_0.1_224_224_chunk21_gripper_len101_hist40/metadata.csv"
]

all_dfs = []
for csv_file in csv_files:
    df = pd.read_csv(csv_file)
    all_dfs.append(df)

combined_df = pd.concat(all_dfs, ignore_index=True)

import datetime
from pathlib import Path
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
metadata_path = Path("/net/holy-isilon/ifs/rc_labs/ydu_lab/xczhang/workspace/SAILOR/env_repos/LIBERO/libero/datasets/metadata")
combined_csv_path = metadata_path / f"metadata_{timestamp}.csv"

combined_df = combined_df.sample(frac=1).reset_index(drop=True)

print(combined_df['split'].value_counts())
print(combined_df['width'].unique())
print(combined_df['height'].unique())

split
training      31956
validation      173
Name: count, dtype: int64
[448]
[448]


In [5]:
def truncate_df(df, validation_size=None, training_size=None):
    validation_df = df[df['split'] == 'validation'].reset_index(drop=True)
    training_df = df[df['split'] == 'training'].reset_index(drop=True)
    
    if validation_size is not None:
        validation_df = validation_df.sample(n=validation_size, random_state=42)
    if training_size is not None:
        training_df = training_df.sample(n=training_size, random_state=42)
    
    truncated_df = pd.concat([validation_df, training_df]).reset_index(drop=True)
    truncated_df = truncated_df.sample(frac=1, random_state=42).reset_index(drop=True)
    print(truncated_df['split'].value_counts())
    print(truncated_df['width'].unique())
    print(truncated_df['height'].unique())
    print(truncated_df['n_frames'].value_counts())
    return truncated_df

In [6]:
truncated_df = truncate_df(check_df, validation_size=128, training_size=None)
truncated_df.to_csv(metadata_csv_path, index=False, quoting=csv.QUOTE_MINIMAL)

split
validation    128
Name: count, dtype: int64
[448]
[448]
n_frames
101    128
Name: count, dtype: int64


In [6]:
truncated_df = truncate_df(combined_df, validation_size=128, training_size=None)

split
training      31956
validation      128
Name: count, dtype: int64
[448]
[448]
n_frames
81     31853
101      128
78        11
74        10
73         9
75         8
77         8
71         7
80         7
67         6
79         6
66         5
72         5
70         5
69         4
76         4
65         3
68         2
61         1
58         1
64         1
Name: count, dtype: int64


In [7]:
def update_metadata(metadata_path):
    df = pd.read_csv(metadata_path)
    for idx, row in df.iterrows():
        if not os.path.exists(row['video_path']):
            df.drop(idx, inplace=True)
    print(df.split.value_counts())
    df.to_csv(metadata_path, index=False)

In [9]:
update_metadata(combined_csv_path)

split
training      30052
validation      173
Name: count, dtype: int64


In [8]:
import csv
print(f"Saving combined metadata to {combined_csv_path}")
combined_df.to_csv(combined_csv_path, index=False, quoting=csv.QUOTE_MINIMAL)

Saving combined metadata to /net/holy-isilon/ifs/rc_labs/ydu_lab/xczhang/workspace/SAILOR/env_repos/LIBERO/libero/datasets/metadata/metadata_20260308_221042.csv


In [ ]:
import pandas as pd
csv_file = "/n/holylabs/ydu_lab/Lab/zhangxiangcheng/code/SAILOR/env_repos/LIBERO/libero/datasets/metadata/metadata_20260218_065837.csv"
df = pd.read_csv(csv_file)
df = df.drop(columns=['prompt_embed_path', 'negative_prompt_embed_path'], errors='ignore')
df.to_csv(csv_file, index=False)
print(f"Updated metadata saved to {csv_file}")

NameError: name 'csv' is not defined